In [1]:
import pandas as pd
import sqlite3

print("Pandas works!")

Pandas works!


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/chess.db')

In [3]:
query = """
SELECT COUNT(*) AS total_games,
       SUM(CASE WHEN rated = 1 THEN 1 ELSE 0 END) AS rated_games
FROM games;
"""

pd.read_sql_query(query, conn)

,total_games,rated_games
0,20058,16155


In [4]:
query = """
SELECT victory_status,
       COUNT(*) AS total
FROM games
GROUP BY victory_status;
"""

pd.read_sql_query(query, conn)

,victory_status,total
0,Draw,906
1,Mate,6325
2,Out of Time,1680
3,Resign,11147


In [6]:
query = """
SELECT game_id, winner, turns
FROM games
ORDER BY turns DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

print(df)


   game_id winner  turns
0    11555  White    349
1    13860  White    349
2    16387   Draw    259
3     4237   Draw    255
4    16646   Draw    226
5    15479   Draw    222
6    16944  Black    222
7     6777   Draw    221
8    13231  Black    218
9    13556   Draw    216


In [8]:
query = """
SELECT 
    winner,
    COUNT(*) * 100.0 / (SELECT COUNT(*) FROM games) AS win_rate_percent
FROM games
GROUP BY winner;
"""

df = pd.read_sql_query(query, conn)

print(df)

  winner  win_rate_percent
0  Black         45.403330
1   Draw          4.736265
2  White         49.860405


In [9]:
query = """
SELECT 
    victory_status,
    AVG(turns) AS avg_turns,
    MAX(turns) AS max_turns
FROM games
GROUP BY victory_status
ORDER BY avg_turns DESC;
"""

df = pd.read_sql_query(query, conn)

print(df)

  victory_status  avg_turns  max_turns
0           Draw  83.781457        259
1    Out of Time  72.742857        349
2           Mate  65.415020        222
3         Resign  53.912533        218


In [10]:
query = """
SELECT 
    victory_status,
    AVG(turns) AS avg_turns,
    MAX(turns) AS max_turns
FROM games
GROUP BY victory_status
ORDER BY avg_turns DESC;
"""

df = pd.read_sql_query(query, conn)

print(df)

  victory_status  avg_turns  max_turns
0           Draw  83.781457        259
1    Out of Time  72.742857        349
2           Mate  65.415020        222
3         Resign  53.912533        218


In [11]:
query = """
SELECT 
    opening_code,
    COUNT(*) AS total_games
FROM games
GROUP BY opening_code
HAVING COUNT(*) > 500
ORDER BY total_games DESC
LIMIT 5;
"""

df = pd.read_sql_query(query, conn)

print(df)

  opening_code  total_games
0          A00         1007
1          C00          844
2          D00          739
3          B01          716
4          C41          691


In [17]:
import sqlite3

conn = sqlite3.connect("data/chess.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('games',)]


In [19]:
query = """
SELECT 
    opening_code,
    opening_fullname,
    COUNT(*) AS total_games
FROM games
GROUP BY opening_code, opening_fullname
ORDER BY total_games DESC
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)

print(df)

  opening_code                               opening_fullname  total_games
0          A00                           Van't Kruijs Opening          368
1          B20               Sicilian Defense: Bowdler Attack          296
2          C00               French Defense: Knight Variation          271
3          B01  Scandinavian Defense: Mieses-Kotroc Variation          259
4          D00                Queen's Pawn Game: Mason Attack          232


In [31]:
import pandas as pd
import sqlite3

# قراءة ملف اللاعبين من CSV
players_df = pd.read_csv("players_registry.csv")

# الاتصال بقاعدة البيانات
conn = sqlite3.connect("chess.db")

# تحويل CSV إلى جدول داخل قاعدة البيانات
players_df.to_sql("players_registry", conn, if_exists="replace", index=False)

215

In [36]:
import pandas as pd
import sqlite3

# قراءة ملف اللاعبين من CSV
players_df = pd.read_csv("chess_games.csv")

# الاتصال بقاعدة البيانات
conn = sqlite3.connect("chess.db")

# تحويل CSV إلى جدول داخل قاعدة البيانات
players_df.to_sql("chess_games", conn, if_exists="replace", index=False)

20058

In [38]:
query = """
SELECT p.*
FROM players_registry p
LEFT JOIN chess_games g
    ON p.username = g.white_id
WHERE g.white_id IS NULL;
"""

df = pd.read_sql_query(query, conn)
print(df)

           username     display_name             country  registered_year  \
0   fandm-lancaste_  Fandm Lancaster              Brazil           2016.0   
1           docbos_          Docboss  russian federation           2020.0   
2          gjdgjdf_         Gjdgjdfj              Brazil           2021.0   
3         e_boecha_        E Boechat              Poland           2016.0   
4            amana_           Amanan                  UA           2019.0   
5   solidchess_heh_  Solidchess Hehe              Russia           2022.0   
6            jg201_           Jg2017               Spain           2016.0   
7           die_uh_          Die Uhr              Poland           2017.0   
8     bigbrother10_    Bigbrother101              Brazil           2021.0   
9             mash_            Mashi             Ukraine           2019.0   
10         davidc8_         Davidc87               Spain           2017.0   
11        mobin138_        Mobin1388               Spain           2015.0   

In [39]:
query = """
WITH player_wins AS (
    SELECT white_id AS username
    FROM chess_games
    WHERE winner = 'White'

    UNION ALL

    SELECT black_id AS username
    FROM chess_games
    WHERE winner = 'Black'
)
SELECT 
    p.display_name,
    p.country,
    COUNT(*) AS total_wins
FROM player_wins w
LEFT JOIN players_registry p
    ON w.username = p.username
GROUP BY w.username
ORDER BY total_wins DESC
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df)

         display_name         country  total_wins
0             Taranga              UA          72
1  Vladimir Kramnik 1  united kingdom          50
2       A P T E M U U             BRA          46
3           Chesscarl              UA          45
4        Ducksandcats           Spain          43


In [41]:
query = """
WITH player_wins AS (
    SELECT white_id AS username FROM chess_games WHERE winner = 'White'
    UNION ALL
    SELECT black_id AS username FROM chess_games WHERE winner = 'Black'
)
SELECT 
    p.username,
    p.display_name,
    p.rating_registry,
    COUNT(*) AS wins
FROM player_wins w
JOIN players_registry p
    ON w.username = p.username
GROUP BY p.username
ORDER BY wins DESC;
"""
df = pd.read_sql_query(query, conn)
print(df)

               username        display_name  rating_registry  wins
0               taranga             Taranga             1431    72
1    vladimir-kramnik-1  Vladimir Kramnik 1             1573    50
2         a_p_t_e_m_u_u       A P T E M U U             1264    46
3             chesscarl           Chesscarl             2091    45
4          ducksandcats        Ducksandcats             1832    43
..                  ...                 ...              ...   ...
195           joe-brown           Joe Brown             1196     7
196        mrmandelbrot        Mrmandelbrot             1202     6
197        element4life        Element4Life             1292     6
198            nitsua49            Nitsua49             1624     5
199              marigw              Marigw              943     5

[200 rows x 4 columns]


In [42]:
query = """
WITH player_wins AS (
    SELECT white_id AS username FROM chess_games WHERE winner = 'White'
    UNION ALL
    SELECT black_id AS username FROM chess_games WHERE winner = 'Black'
)
SELECT 
    p.display_name,
    p.rating_registry,
    COUNT(*) AS wins
FROM player_wins w
JOIN players_registry p
    ON w.username = p.username
GROUP BY p.username
ORDER BY p.rating_registry DESC;
"""
df = pd.read_sql_query(query, conn)
print(df)

          display_name  rating_registry  wins
0            Lance5500             2621    33
1          Oso Bucholz             2290    13
2           Bosspotato             2287    29
3         Youralterego             2222    31
4           Doraemon61             2218    38
..                 ...              ...   ...
195      Businessman47             1131    18
196   Iamaliveandbored             1127    16
197          Andreas00             1111    10
198  Shivangithegenius              978    11
199             Marigw              943     5

[200 rows x 3 columns]


In [43]:
query = """
WITH player_wins AS (
    SELECT white_id AS username FROM chess_games WHERE winner = 'White'
    UNION ALL
    SELECT black_id AS username FROM chess_games WHERE winner = 'Black'
)
SELECT 
    p.display_name,
    p.rating_registry,
    COUNT(*) AS wins
FROM player_wins w
JOIN players_registry p
    ON w.username = p.username
GROUP BY p.username
ORDER BY p.rating_registry DESC;
"""
df = pd.read_sql_query(query, conn)
print(df)

          display_name  rating_registry  wins
0            Lance5500             2621    33
1          Oso Bucholz             2290    13
2           Bosspotato             2287    29
3         Youralterego             2222    31
4           Doraemon61             2218    38
..                 ...              ...   ...
195      Businessman47             1131    18
196   Iamaliveandbored             1127    16
197          Andreas00             1111    10
198  Shivangithegenius              978    11
199             Marigw              943     5

[200 rows x 3 columns]


In [44]:
query = """
SELECT 
    game_id,
    white_id,
    white_rating,
    LAG(white_rating) OVER (
        PARTITION BY white_id
        ORDER BY game_id
    ) AS previous_white_rating
FROM chess_games
ORDER BY white_id, game_id;
"""
df = pd.read_sql_query(query, conn)
print(df)

       game_id             white_id  white_rating  previous_white_rating
0        10139              --jim--           986                    NaN
1         9788  -l-_jedi_knight_-l-          1564                    NaN
2         9789  -l-_jedi_knight_-l-          1465                 1564.0
3         9791  -l-_jedi_knight_-l-          1473                 1465.0
4         9793  -l-_jedi_knight_-l-          1486                 1473.0
...        ...                  ...           ...                    ...
20053    10401                zynko          1494                 1515.0
20054     4722              zzeecco          1525                    NaN
20055    17215            zztopillo          1466                    NaN
20056    14972               zzzbbb          1616                    NaN
20057    19205              zzzimon          1375                    NaN

[20058 rows x 4 columns]
